# เทรนขยะ 4 ประเภท — ชุดข้อมูล 200 ภาพ (Hazardous Validation วัตถุเดียว)
200 ภาพ / 229 กรอบ: Train 151, Val 25, Test 24
กรอบใหม่เป็นร่างให้ตรวจยืนยัน ของมีคมรวมถึงกรรไกรเป็น Hazardous ตามกติกาโครงงานนี้
รูปใหม่ทั้งหมดอยู่ใน train และยังไม่มี test กรรไกรจากกล้องจริง


Validation เวอร์ชันนี้เปลี่ยนเฉพาะ Hazardous ให้เป็นภาพวัตถุเดียว 5 ภาพ คะแนนเทียบตรงกับ Validation เดิมไม่ได้ ส่วน Train และ Test คงเดิม


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.160'])
import torch
assert torch.cuda.is_available(), 'กรุณาเปิด GPU ใน Settings > Accelerator ก่อนรันต่อ'
print('GPU:', torch.cuda.get_device_name(0))


## 1. เลือกชุดข้อมูล v2
เพิ่ม ZIP ชุดใหม่เป็น Dataset ใน Input แล้วรันต่อได้เลย ระบบค้นหา datasetreal-v3-hazard-single.marker ให้อัตโนมัติ ไม่ต้องแก้ path และไม่เลือกชุด 40 ภาพเดิม


In [ ]:
from pathlib import Path
for path in Path('/kaggle/input').rglob('*.yaml'):
    if path.name in ('data.yaml', 'dataset.yaml'):
        print(path)


In [ ]:
import yaml
candidates = [p.parent / 'data.yaml' for p in Path('/kaggle/input').rglob('datasetreal-v3-hazard-single.marker')]
assert len(candidates) == 1, f'ต้องเพิ่ม Dataset v2 หนึ่งชุด; พบ {candidates}'
SOURCE_YAML = candidates[0]
# โฟลเดอร์ที่มี images/labels หรือ train/valid/test (แก้ได้หาก YAML อยู่คนละระดับ)
DATA_ROOT = SOURCE_YAML.parent
# ถ้า YAML ใช้ path แบบเก่าจากเครื่องอื่น ให้ระบุโฟลเดอร์ภาพตรงนี้
SPLIT_PATHS = {}  # เช่น {'train':'train/images', 'val':'valid/images', 'test':'test/images'}
NAMES = ['Recyclable', 'Organic', 'General', 'Hazardous']
assert SOURCE_YAML.is_file(), 'แก้ SOURCE_YAML ให้ตรงกับไฟล์ Input ก่อน'
original = yaml.safe_load(SOURCE_YAML.read_text())
names = original.get('names')
if isinstance(names, dict):
    assert sorted(names) == list(range(4)), 'Class IDs ต้องเป็น 0,1,2,3'
    names = [names[i] for i in range(4)]
assert names == NAMES, f'ลำดับคลาสไม่ตรง: {names}; ต้องตรวจและ remap labels ก่อน'

def resolve_split(split):
    value = SPLIT_PATHS.get(split, original.get(split))
    assert isinstance(value, str) and value, f'ต้องระบุโฟลเดอร์ภาพ {split}'
    p = Path(value)
    if p.is_absolute() and p.is_dir():
        return p.resolve()
    bases = [DATA_ROOT]
    yaml_root = original.get('path')
    if isinstance(yaml_root, str):
        bases.append(SOURCE_YAML.parent / yaml_root)
    candidates = []
    for base in bases:
        candidate = (base / value).resolve()
        if candidate.is_dir() and candidate not in candidates:
            candidates.append(candidate)
    assert len(candidates) == 1, f'หา {split} ไม่เจอหรือกำกวม ให้กำหนด SPLIT_PATHS: {value}'
    return candidates[0]

splits = {key: resolve_split(key) for key in ('train', 'val', 'test')}
config = {'path': str(DATA_ROOT.resolve()), **{k:str(v) for k,v in splits.items()}, 'names':dict(enumerate(NAMES))}
WORK = Path('/kaggle/working')
WORK.mkdir(exist_ok=True)
DATA_YAML = WORK / 'waste-data.yaml'
DATA_YAML.write_text(yaml.safe_dump(config, sort_keys=False))
print(DATA_YAML.read_text())


## 2. ตรวจภาพและ labels
ตรวจไฟล์คู่กัน ขอบเขตกรอบ จำนวนคลาส และภาพซ้ำข้ามชุด (ภาพคล้ายกันต้องตรวจเองเพิ่มเติม)

In [ ]:
import hashlib, math
from collections import Counter
from PIL import Image
hashes = {}
errors = []
for split, directory in splits.items():
    images = sorted(p for p in directory.rglob('*') if p.suffix.lower() in ('.jpg','.jpeg','.png','.bmp'))
    assert images, f'{split} ไม่มีภาพ'
    counts = Counter()
    for image in images:
        try:
            with Image.open(image) as im:
                im.verify()
            digest = hashlib.sha256(image.read_bytes()).hexdigest()
            if digest in hashes and hashes[digest] != split:
                errors.append(f'ภาพซ้ำข้ามชุด: {image}')
            hashes[digest] = split
            parts = list(image.parts)
            assert 'images' in parts, 'ภาพต้องอยู่ใต้โฟลเดอร์ images'
            index = len(parts) - 1 - parts[::-1].index('images')
            parts[index] = 'labels'
            label = Path(*parts).with_suffix('.txt')
            assert label.is_file(), f'ไม่มี label: {label}'
            for line in label.read_text().splitlines():
                if not line.strip():
                    continue
                c,x,y,w,h = map(float, line.split())
                assert all(math.isfinite(v) for v in (c,x,y,w,h))
                assert c in range(4) and 0 < w <= 1 and 0 < h <= 1
                assert x-w/2 >= -0.001 and y-h/2 >= -0.001 and x+w/2 <= 1.001 and y+h/2 <= 1.001
                counts[int(c)] += 1
        except Exception as exc:
            errors.append(f'{image}: {exc or "label ไม่ถูกต้อง"}')
    print(split, len(images), 'images', {NAMES[i]: counts[i] for i in range(4)})
    if any(counts[i] == 0 for i in range(4)):
        errors.append(f'{split} มีคลาสที่ไม่มี bounding box กรุณาจัดชุดข้อมูลให้ครบทั้ง 4 ประเภท')
assert not errors, '\n'.join(errors[:30])
print('Dataset check passed')


## 3. เทรน
ตั้งไว้สูงสุด 100 epochs และหยุดเมื่อผล validation ไม่ดีขึ้น 25 รอบ ใช้ batch 8 และภาพ 640 พิกเซล ถ้า GPU หน่วยความจำไม่พอให้ลด BATCH เป็น 4
จำนวนรอบไม่รับประกันความแม่นยำ ให้ตรวจคะแนนและภาพผลทำนายหลังเทรน


In [ ]:
from ultralytics import YOLO
EPOCHS = 100
BATCH = 8
# Start from pretrained general-object weights, not the weak previous waste model.
model = YOLO('yolov8n.pt')
model.train(data=str(DATA_YAML), epochs=EPOCHS, imgsz=640, batch=BATCH,
            device=0, workers=2, seed=42, patience=25, cache=False,
            project=str(WORK / 'runs'), name='waste_real_v3_hazard_single', plots=True)
RUN_DIR = Path(model.trainer.save_dir)
BEST = RUN_DIR / 'weights/best.pt'
assert BEST.is_file()
print('Best weights:', BEST)


## 4. ประเมินกับ Test
ใช้ Test หลังเลือกโมเดลด้วย Validation แล้ว ไม่ควรปรับโมเดลซ้ำตามคะแนน Test

In [ ]:
import json
model = YOLO(str(BEST))
for split, filename in [('val', 'validation-metrics.json'), ('test', 'test-metrics.json')]:
    metrics = model.val(data=str(DATA_YAML), split=split, device=0,
                        project=str(WORK / 'evaluation'), name=split, plots=True)
    summary = {key: float(value) for key, value in metrics.results_dict.items()}
    (WORK / filename).write_text(json.dumps(summary, indent=2))
    print(split, summary)

# Show every test image at the same confidence threshold as the website.
from PIL import Image
from IPython.display import display
results = model.predict(source=str(splits['test']), conf=0.45, imgsz=640,
                        device=0, save=True, project=str(WORK), name='evaluation_preview')
report = []
for result in results:
    print(Path(result.path).name, 'detections:', len(result.boxes))
    preview = Image.fromarray(result.plot()[..., ::-1])
    preview.thumbnail((640, 480))
    display(preview)
    report.append({'image': Path(result.path).name, 'detections': [
        {'class': model.names[int(b.cls.item())], 'confidence': float(b.conf.item())}
        for b in result.boxes]})
(WORK / 'predictions-at-045.json').write_text(json.dumps(report, indent=2))


## ดาวน์โหลดผล
ใช้แถบ Output ด้านขวา → /kaggle/working → เมนูไฟล์ → Download (ไม่ใช้ลิงก์ FileLink ที่เคยขึ้น 404)
เก็บผลก่อนปิด session และบันทึกเวอร์ชันพร้อม outputs


In [ ]:
import shutil
from zipfile import ZipFile, ZIP_DEFLATED
shutil.copy2(BEST, WORK / 'best.pt')
last = RUN_DIR / 'weights/last.pt'
if last.is_file(): shutil.copy2(last, WORK / 'last.pt')
(WORK / 'requirements-training.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))
items = [WORK / name for name in ['best.pt','validation-metrics.json','test-metrics.json',
                                'predictions-at-045.json','requirements-training.txt']]
items += [p for p in RUN_DIR.rglob('*') if p.is_file() and p.suffix != '.pt']
items += [p for p in (WORK / 'evaluation').rglob('*') if p.is_file()]
items += [p for p in (WORK / 'evaluation_preview').rglob('*') if p.is_file()]
with ZipFile(WORK / 'waste-real-v2-results.zip', 'w', ZIP_DEFLATED) as z:
    for p in items:
        if p.is_file(): z.write(p, p.relative_to(WORK))
print('Download from Output: best.pt and waste-real-v2-results.zip')


## 6. นำกลับไปใช้บนเครื่อง
วาง best.pt ในโฟลเดอร์โปรเจกต์ แล้วรันใน PowerShell:

```powershell
.\.venv\Scripts\python.exe web_app.py --weights best.pt
```
เลือกโมเดล **ขยะ 4 ประเภท** และแหล่งภาพ **กล้อง Webcam** บนเว็บไซต์
Notebook นี้เทรนตัวตรวจจับ YOLO ส่วน ByteTrack ในโปรเจกต์ไม่ต้องเทรนแยก

แหล่งอ้างอิง: https://www.kaggle.com/docs/notebooks และ https://docs.ultralytics.com/modes/train/
